# Radiology report template-editing pipeline

Converts short radiologist dictation into a completed structured report by editing
the supplied normal `template_content`, using the Anthropic API (Claude).

This notebook is self-contained and reproduces `submission.csv` end-to-end:
1. Reads `train.csv` / `test.csv`.
2. Builds a prompt from the calibrated rules in `RULES.md` (embedded below, verbatim).
3. Calls the Anthropic API once per case, validates the output format, retries on
   malformed responses.
4. Self-validates on a `train.csv` sample using a local approximate re-implementation
   of the RES scorer, so the pipeline reports an expected-quality signal before it
   ever touches the test set.
5. Writes `submission.csv` with exactly `case_id,report`.

**No API key is stored in this notebook.** Set it as a Kaggle Secret named
`ANTHROPIC_API_KEY` (Add-ons -> Secrets) or as an environment variable before running.


In [ ]:
!pip install -q anthropic


In [ ]:
import os
import re
import csv
import json
import time
import unicodedata
from dataclasses import dataclass

import pandas as pd
import anthropic


## Configuration


In [ ]:
# Adjust these paths to your Kaggle dataset mount, e.g. /kaggle/input/<competition>/
TRAIN_CSV = "train.csv"
TEST_CSV = "test.csv"
SAMPLE_SUBMISSION_CSV = "sample_submission.csv"
OUTPUT_CSV = "submission.csv"

# "claude-sonnet-5" is the default (best cost/quality balance for this task).
# Set to "claude-opus-5" for the highest-quality option if available on your account.
MODEL = os.environ.get("REPORT_MODEL", "claude-sonnet-5")

MAX_RETRIES = 3
REQUEST_TIMEOUT = 120
SLEEP_BETWEEN_CALLS = 0.5  # be polite to the rate limiter

N_CALIBRATION_SAMPLES = 20  # how many train.csv rows to self-validate on before test generation


## API key

Never hardcode a key. On Kaggle, add it as a Secret (Add-ons -> Secrets) named
`ANTHROPIC_API_KEY` and attach it to the notebook; it's then exposed via
`kaggle_secrets`. Locally, just export `ANTHROPIC_API_KEY` in your shell before
launching Jupyter.


In [ ]:
def get_api_key() -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    except Exception:
        pass
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError(
            "Set ANTHROPIC_API_KEY as a Kaggle Secret or environment variable before running."
        )
    return key

client = anthropic.Anthropic(api_key=get_api_key())


## Calibrated rules (embedded verbatim from `RULES.md`)

This is the exact spec used to hand-generate the submitted reports, kept in one
file (`RULES.md`) and embedded here so the API prompt and the manual generation
process can never drift apart.


In [ ]:
RULES_MD = "# Template-editing rules (shared spec)\n\nThis is the single source of truth for turning `(modality, body_part, study_description,\npatient_age_band, patient_sex, template_content, dictation)` into a completed report. It is\nused verbatim by:\n\n1. Manual/agentic generation (Claude Code reasoning through cases directly), and\n2. The Kaggle notebook's API prompt (`notebook/pipeline.ipynb`).\n\nKeeping one wording for both means the notebook's output should closely match what was\ngenerated manually, which matters for the \"must reproduce the uploaded CSV without manual\ncase-level editing\" requirement.\n\n## Output contract\n\nReturn **only**:\n\n```\nFINDINGS:\n<fields, in the same order and with the same field labels as template_content>\n\nIMPRESSION:\n<concise summary>\n```\n\nNo preamble, no explanation, no markdown fences, no extra sections. Do not invent a field\nlabel that isn't in `template_content`. Do not drop a field label that is in\n`template_content` \u2014 if the dictation says nothing about it, keep the template's normal\nsentence for that field unchanged.\n\n## Formatting rules (mechanical, apply every time)\n\n- **Field labels are always rendered in ALL CAPS** in the output, even if `template_content`\n  wrote them in Title Case or mixed case (e.g. template `Hip Joints:` \u2192 output\n  `HIP JOINTS:`; template `Medial meniscus:` \u2192 output `MEDIAL MENISCUS:`). This is a\n  systematic transformation applied to every field label, independent of content changes.\n- **Only emit `FINDINGS:` and `IMPRESSION:`.** Some templates carry extra boilerplate after\n  the normal fields \u2014 a legend/menu of categories (e.g. a full Lung-RADS category\n  reference table), a `RECOMMENDATION:` placeholder, blank `[]` placeholders, a scoring\n  key, version/date footer text. None of that is a field to preserve \u2014 drop it entirely.\n  Only real anatomic/system FINDINGS fields and the two required section headers survive\n  into the output.\n- If the dictation already supplies a clean, well-organized impression (a short list of\n  numbered/plain conclusions), reuse it close to verbatim rather than rewriting it in your\n  own words \u2014 don't paraphrase good dictated prose for its own sake.\n\n## Core rules\n\n1. **The template is the starting report, not an example.** Start from\n   `template_content` and edit it \u2014 don't write a report from scratch.\n2. **Route every dictated finding to the matching FINDINGS field.** Read the dictation\n   fully first, then decide which template field each finding belongs to by anatomy/topic,\n   not by the order it was dictated in. A finding about the pleura goes under the pleura\n   field even if it was dictated before a lung finding.\n3. **Replace or modify the normal statement when the dictation contradicts it.** If a field's\n   template sentence is a normal/negative statement (\"no acute fracture\", \"tendon is intact\")\n   and the dictation describes an abnormality in that anatomic area, edit that sentence to\n   state the abnormality. Keep any part of the template sentence that is still true (e.g. if\n   only one of two structures in a combined field is abnormal, keep the negative statement\n   for the structure that's still normal).\n4. **Preserve untouched fields verbatim \u2014 but \"untouched\" means the dictation said\n   nothing about that anatomy, not \"the finding is normal\".** If the dictation says\n   nothing that bears on a field, leave that field's template wording exactly as-is: don't\n   paraphrase, reorder, or \"polish\" it.\n   **However**, if the dictation *does* narrate that anatomy \u2014 even just to confirm it's\n   normal, using its own specific wording (naming the exact vessels/structures examined,\n   phrases like \"where visualized\", \"grossly clear\", extra qualifiers) \u2014 use the\n   dictation's phrasing for that field instead of the template's generic sentence, even\n   though the clinical conclusion (still normal) hasn't changed. Reference reports\n   consistently rewrite normal fields to match a detailed dictation's own words rather than\n   leaving the boilerplate in place; only fields the dictation is silent on keep the exact\n   template sentence. A terse/blanket dictation (\"normal\", \"normal chest\") with no\n   per-field detail gives nothing to substitute, so every field then stays as template text.\n5. **Do not add unsupported content.** Every clinical statement in the output must trace to\n   either the dictation or the template's own normal wording. Never introduce a finding,\n   measurement, or diagnosis that isn't in one of those two sources. Vitals like \"thank you\n   for your referral\", \"correlate clinically\" instructions, or dictation preamble about the\n   technique/history are not findings \u2014 leave them out unless the template already has a\n   place for exam-quality/technique remarks.\n6. **Route every dictated finding somewhere, even ones with no matching field \u2014 normal\n   findings included.** If the template has an `OTHER FINDINGS:` field, put any dictated\n   finding (abnormal *or* normal/negative) that doesn't cleanly belong to another labelled\n   field there. Leave it empty (no content after the label) only if truly nothing applies \u2014\n   never delete the label itself. If the template has **no** `OTHER FINDINGS:` field and a\n   dictated finding still has no matching labelled field (e.g. the dictation describes an\n   organ system the template doesn't break out, like incidental brain findings on a\n   pituitary-focused MRI, or visualized nerves on a joint MRI), add it as an unlabelled\n   paragraph inside the FINDINGS section \u2014 wherever it reads naturally, typically right\n   after the last labelled field (or, for exam-quality/technique remarks, before the first\n   field). Do not silently drop a dictated finding just because no field name fits it.\n7. **Resolve placeholders.** Templates sometimes contain bracket placeholders such as\n   `[left/right]`, `[_laterality_]`, `[normal/abnormal]`. Replace these with the actual value\n   supported by the dictation (e.g. \"either lower extremity\" for bilateral, \"the right\n   shoulder\" for unilateral-right). Never leave literal brackets in the output.\n8. **Laterality and negation are high-value tokens \u2014 get them exactly right.** \"No pleural\n   effusion\" vs \"small right pleural effusion\" is the entire point of the exercise; a wrong\n   negation or wrong side is penalized heavily by the scorer's critical-word weighting.\n9. **Preserve template wording and field order whenever possible.** When you do need to\n   state an abnormality, prefer editing the existing template sentence (swap the normal\n   claim for the abnormal one, keep the surrounding phrasing/structure) over rewriting the\n   whole field in new prose. Keep the fields in the same top-to-bottom order as\n   `template_content`.\n10. **IMPRESSION is a concise summary, not a repeat of FINDINGS.** State only the\n    clinically important abnormal findings (typically as a short numbered list when there\n    are multiple), optionally with a supported differential/likely diagnosis if the FINDINGS\n    (dictation) explicitly suggests one. Do not restate every normal field. If nothing\n    abnormal was dictated, keep the template's normal IMPRESSION statement (resolving any\n    placeholder, e.g. \"Normal MRI of the right shoulder.\").\n    - A recommendation/follow-up instruction dictated alongside a finding (e.g. \"advise MRI\n      for further evaluation\", \"clinical correlation advised\") is impression-level content \u2014\n      state it in IMPRESSION, not inside the FINDINGS field for that finding.\n    - Even when abnormal findings are added, it's common to keep a final residual negative\n      line adapted from the template's own default IMPRESSION (e.g. \"No acute fracture or\n      dislocation.\", \"No acute cardiopulmonary abnormality.\") as the last bullet, scoped to\n      what's still true (an incidental non-acute lesion doesn't preclude \"no acute fracture\").\n      Only drop it if the dictation's own summary already covers the same ground or the\n      finding contradicts it.\n11. **Garbled/misspelled dictation:** dictation is raw transcription and may contain typos\n    (\"effusuon\"), shorthand, or fragments (\"no effusion, infiltrates\"). Interpret intent\n    conservatively and route to the right field; do not copy typos into the report.\n12. **modality / body_part / study_description / patient_age_band / patient_sex are context\n    only.** Use them only to disambiguate pronouns/laterality/anatomy \u2014 never invent findings\n    from them (e.g. don't add age-related degenerative findings just because the patient is\n    elderly, unless dictation or template says so).\n\n## Worked example\n\nDictation: `mild right basilar opacity, small right pleural effusion`\n\nTemplate:\n```\nFINDINGS:\nSUPPORT DEVICES: None.\nCARDIOMEDIASTINAL SILHOUETTE: Within normal size limits.\nLUNGS: No focal airspace opacity or pulmonary edema.\nPLEURA: No pleural effusion or pneumothorax.\nOSSEOUS STRUCTURES: No acute osseous abnormality identified on these views.\nIMPRESSION:\nNo acute cardiopulmonary abnormality.\n```\n\nExpected:\n```\nFINDINGS:\nSUPPORT DEVICES: None.\nCARDIOMEDIASTINAL SILHOUETTE: Within normal size limits.\nLUNGS: Mild right basilar airspace opacity. No pulmonary edema.\nPLEURA: Small right pleural effusion. No pneumothorax.\nOSSEOUS STRUCTURES: No acute osseous abnormality identified on these views.\nIMPRESSION:\nMild right basilar airspace opacity and small right pleural effusion.\n```\n\nNote the opacity is not placed under PLEURA and the effusion is not placed under LUNGS \u2014\nfield routing by anatomy, not dictation order, is the whole game.\n\n## Self-check before finalizing a case\n\n- [ ] Every FINDINGS field label from `template_content` is present, same order.\n- [ ] Every dictated finding appears under the anatomically correct field (or OTHER FINDINGS\n      if no better field exists).\n- [ ] Untouched fields are byte-for-byte the template wording.\n- [ ] No placeholder brackets remain.\n- [ ] No fact appears that isn't traceable to dictation or template.\n- [ ] Negation/laterality/measurements match the dictation exactly.\n- [ ] IMPRESSION reflects only the abnormal findings (or the template's normal line if\n      nothing abnormal was dictated), with no new information.\n"


In [ ]:
OUTPUT_CONTRACT_REMINDER = (
    "Return ONLY the completed report text, starting with 'FINDINGS:' and containing "
    "exactly one 'IMPRESSION:' section. No preamble, no explanation, no markdown code "
    "fences, no other headings."
)

SYSTEM_PROMPT = RULES_MD + "\n\n" + OUTPUT_CONTRACT_REMINDER


## Prompt construction


In [ ]:
def build_user_prompt(row) -> str:
    return (
        f"modality: {row['modality']}\n"
        f"body_part: {row['body_part']}\n"
        f"study_description: {row['study_description']}\n"
        f"patient_age_band: {row['patient_age_band']}\n"
        f"patient_sex: {row['patient_sex']}\n\n"
        f"template_content:\n{row['template_content']}\n\n"
        f"dictation:\n{row['dictation']}\n\n"
        "Produce the completed report now, following the rules exactly."
    )


## Generation with format validation + retry

A valid response must start with `FINDINGS:` and contain exactly one `IMPRESSION:`
section. If the model returns something malformed (preamble, markdown fences, missing
section), we send one corrective follow-up before giving up.


In [ ]:
VALID_RE = re.compile(r"^\s*FINDINGS:", re.IGNORECASE)
IMPRESSION_RE = re.compile(r"IMPRESSION\s*:", re.IGNORECASE)


def is_valid_report(text: str) -> bool:
    if not text:
        return False
    if not VALID_RE.match(text.strip()):
        return False
    if len(IMPRESSION_RE.findall(text)) != 1:
        return False
    return True


def clean_report(text: str) -> str:
    text = text.strip()
    # Strip accidental markdown code fences.
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
    text = re.sub(r"\n?```$", "", text)
    return text.strip()


def generate_report(row) -> str:
    messages = [{"role": "user", "content": build_user_prompt(row)}]
    last_text = ""
    for attempt in range(MAX_RETRIES):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            temperature=0,
            system=SYSTEM_PROMPT,
            messages=messages,
        )
        text = clean_report("".join(b.text for b in resp.content if b.type == "text"))
        last_text = text
        if is_valid_report(text):
            return text
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": (
            "That response did not follow the required output contract. "
            "Return ONLY the report, starting with 'FINDINGS:' and containing exactly "
            "one 'IMPRESSION:' section, no other text."
        )})
        time.sleep(SLEEP_BETWEEN_CALLS)
    return last_text  # best effort after MAX_RETRIES


## Local approximate RES scorer (for self-validation only)

Reconstructed from the published scoring rules (text normalization, weighted
word-edit distance, field-aware FINDINGS score, IMPRESSION score). The exact word-weight
lists used by the real Kaggle scorer aren't published, so this is a calibration aid,
not a guaranteed match to the leaderboard number -- but it reliably catches routing,
unsupported-content, and formatting regressions before submission.


In [ ]:
"""
Approximate local implementation of the RES (Radiology Edit Score) described in the
challenge overview, for calibrating our generation approach against train.csv before
generating test predictions.

IMPORTANT: this is our best-effort reconstruction of the published rules (text
normalization, weighted word-edit distance, field-aware FINDINGS score, IMPRESSION
score). The exact word-weight lists and unexpected-field penalty used by the real
Kaggle scorer are not published, so this is a calibration aid to catch regressions
and compare candidate reports directionally -- not a guaranteed match to the
leaderboard number.

Usage as a library:

    from res_scorer import score_report
    result = score_report(template_content, reference_report, candidate_report)
    print(result["RES"], result["F"], result["I"])

CLI:

    python3 res_scorer.py --data data/train.csv --predictions preds.csv
        preds.csv must have columns: case_id,report
        data.csv must have columns: case_id,template_content,report (reference)
"""
import argparse
import csv
import re
import sys
import unicodedata
from dataclasses import dataclass, field

# ---------------------------------------------------------------------------
# Word classification / weights
# ---------------------------------------------------------------------------

NEGATION = {
    "no", "not", "none", "without", "free", "absent", "negative", "unremarkable",
    "normal", "nor",
}
LATERALITY = {
    "left", "right", "bilateral", "bilaterally", "unilateral", "midline",
}
SEVERITY_ACUITY = {
    "mild", "moderate", "severe", "marked", "minimal", "trace", "small", "large",
    "tiny", "massive", "acute", "chronic", "subacute", "stable", "new", "interval",
    "increased", "decreased", "worsening", "improving", "extensive", "focal",
    "diffuse", "high", "low", "grade", "advanced", "early",
}
UNITS = {"mm", "cm", "ml", "cc", "kg", "mg"}

FUNCTION_WORDS = {
    "the", "and", "of", "with", "a", "an", "is", "are", "was", "were", "in", "on",
    "at", "to", "for", "by", "or", "this", "these", "that", "there", "been", "be",
    "as", "which", "from", "it", "its", "than", "into", "over", "under", "within",
    "including", "than", "such",
}

W_CRITICAL = 4.00
W_CONTENT = 2.00
W_FUNCTION = 0.25

UNIT_SPELLINGS = {
    "millimeters": "mm", "millimeter": "mm", "millimetre": "mm", "millimetres": "mm",
    "centimeters": "cm", "centimeter": "cm", "centimetre": "cm", "centimetres": "cm",
    "milliliters": "ml", "milliliter": "ml",
}

NUM_RE = re.compile(r"^[+-]?\d+\.?\d*$")
TOKEN_RE = re.compile(r"[+-]?\d+\.?\d*|[a-z]+")
LEADING_MARKER_RE = re.compile(r"^\s*(?:\d+[\.\)]|[-*•])\s+")
LETTER_HYPHEN_LETTER_RE = re.compile(r"(?<=[a-zA-Z])-(?=[a-zA-Z])")


def token_weight(tok: str) -> float:
    if NUM_RE.match(tok):
        return W_CRITICAL
    if tok in UNITS:
        return W_CRITICAL
    if tok in NEGATION or tok in LATERALITY or tok in SEVERITY_ACUITY:
        return W_CRITICAL
    if tok in FUNCTION_WORDS:
        return W_FUNCTION
    return W_CONTENT


def normalize_and_tokenize(text: str):
    if text is None:
        return []
    text = unicodedata.normalize("NFKC", text).lower()
    lines = text.splitlines()
    cleaned_lines = [LEADING_MARKER_RE.sub("", ln) for ln in lines]
    text = " ".join(cleaned_lines)
    text = LETTER_HYPHEN_LETTER_RE.sub("", text)
    raw_tokens = TOKEN_RE.findall(text)
    tokens = [UNIT_SPELLINGS.get(t, t) for t in raw_tokens]
    return tokens


# ---------------------------------------------------------------------------
# Weighted word-level Levenshtein
# ---------------------------------------------------------------------------

def weighted_edit_cost(ref_tokens, sub_tokens):
    n, m = len(ref_tokens), len(sub_tokens)
    ref_w = [token_weight(t) for t in ref_tokens]
    sub_w = [token_weight(t) for t in sub_tokens]

    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = dp[i - 1][0] + ref_w[i - 1]
    for j in range(1, m + 1):
        dp[0][j] = dp[0][j - 1] + sub_w[j - 1]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_tokens[i - 1] == sub_tokens[j - 1]:
                sub_cost = dp[i - 1][j - 1]
            else:
                sub_cost = dp[i - 1][j - 1] + max(ref_w[i - 1], sub_w[j - 1])
            del_cost = dp[i - 1][j] + ref_w[i - 1]
            ins_cost = dp[i][j - 1] + sub_w[j - 1]
            dp[i][j] = min(sub_cost, del_cost, ins_cost)
    return dp[n][m]


def word_edit_score(reference_text: str, submitted_text: str) -> float:
    ref_tokens = normalize_and_tokenize(reference_text)
    sub_tokens = normalize_and_tokenize(submitted_text)
    if not ref_tokens and not sub_tokens:
        return 0.0
    cost = weighted_edit_cost(ref_tokens, sub_tokens)
    ref_w = sum(token_weight(t) for t in ref_tokens)
    sub_w = sum(token_weight(t) for t in sub_tokens)
    denom = max(ref_w, sub_w)
    if denom == 0:
        return 0.0
    return min(1.0, cost / denom)


# ---------------------------------------------------------------------------
# Report parsing: FINDINGS / IMPRESSION split, field-label extraction
# ---------------------------------------------------------------------------

LABEL_LINE_RE = re.compile(r"^\s*([A-Za-z][A-Za-z0-9 /,.'\-]{1,60}):\s*(.*)$")


def split_sections(report: str):
    """Return (findings_text, impression_text)."""
    if not report:
        return "", ""
    m_f = re.search(r"FINDINGS\s*:", report, re.IGNORECASE)
    m_i = re.search(r"IMPRESSION\s*:", report, re.IGNORECASE)
    if not m_f or not m_i:
        return report, ""
    findings = report[m_f.end():m_i.start()]
    impression = report[m_i.end():]
    return findings.strip(), impression.strip()


def parse_fields(findings_text: str):
    """Return ordered list of (label, content) plus a special '' label for any
    unlabelled preamble content before the first field header."""
    fields = []
    preamble_lines = []
    current_label = None
    current_lines = []

    def flush():
        if current_label is not None:
            fields.append((current_label, "\n".join(current_lines).strip()))

    for line in findings_text.splitlines():
        m = LABEL_LINE_RE.match(line)
        if m:
            flush()
            current_label = m.group(1).strip().upper()
            current_lines = [m.group(2)] if m.group(2) else []
        else:
            if current_label is None:
                preamble_lines.append(line)
            else:
                current_lines.append(line)
    flush()

    preamble = "\n".join(preamble_lines).strip()
    return preamble, fields


# ---------------------------------------------------------------------------
# Field-aware FINDINGS score + IMPRESSION score + RES_case
# ---------------------------------------------------------------------------

CHANGED_EPS = 0.02


def score_report(template_content: str, reference_report: str, candidate_report: str):
    _, template_fields = parse_fields(split_sections(template_content)[0]
                                       if "IMPRESSION" in (template_content or "").upper()
                                       else (template_content or ""))
    # template_content may itself be FINDINGS-only (no IMPRESSION); handle both.
    if not template_fields:
        _, template_fields = parse_fields(template_content or "")
    template_map = {label: content for label, content in template_fields}

    ref_findings, ref_impression = split_sections(reference_report)
    cand_findings, cand_impression = split_sections(candidate_report)

    ref_preamble, ref_fields = parse_fields(ref_findings)
    cand_preamble, cand_fields = parse_fields(cand_findings)
    cand_map = {label: content for label, content in cand_fields}
    cand_labels_seen = set()

    weighted_sum = 0.0
    weight_total = 0.0
    field_details = []

    for label, ref_content in ref_fields:
        template_content_for_label = template_map.get(label, "")
        changed_score = word_edit_score(template_content_for_label, ref_content)
        f_weight = 3.0 if changed_score > CHANGED_EPS else 1.0

        cand_content = cand_map.get(label)
        if cand_content is None:
            edit = word_edit_score(ref_content, "")
            present = False
        else:
            edit = word_edit_score(ref_content, cand_content)
            present = True
            cand_labels_seen.add(label)

        weighted_sum += f_weight * edit
        weight_total += f_weight
        field_details.append({
            "label": label, "field_weight": f_weight, "edit_score": edit,
            "present_in_candidate": present,
        })

    # Unexpected fields present in candidate but not in reference: penalize.
    extra_details = []
    for label, cand_content in cand_fields:
        if label in {l for l, _ in ref_fields}:
            continue
        edit = word_edit_score("", cand_content)
        weighted_sum += 1.0 * edit
        weight_total += 1.0
        extra_details.append({"label": label, "edit_score": edit})

    # Unsupported unlabelled preamble content in candidate not present in reference.
    if cand_preamble and word_edit_score(ref_preamble, cand_preamble) > CHANGED_EPS:
        edit = word_edit_score(ref_preamble, cand_preamble)
        weighted_sum += 1.0 * edit
        weight_total += 1.0

    F = weighted_sum / weight_total if weight_total else 0.0
    I = word_edit_score(ref_impression, cand_impression)
    RES = 0.65 * F + 0.35 * I

    return {
        "RES": RES, "F": F, "I": I,
        "field_details": field_details,
        "extra_fields": extra_details,
    }


# ---------------------------------------------------------------------------


## Step 1: self-validate on a train.csv sample

Runs the exact same `generate_report` pipeline against a sample of `train.csv` (which
has a reference `report` column) and reports the mean approximate RES. This is a
sanity check that the deployed prompt still performs before spending API calls on the
un-scored test set.


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
calib_sample = train_df.sample(n=min(N_CALIBRATION_SAMPLES, len(train_df)), random_state=0)

calib_scores = []
for _, row in calib_sample.iterrows():
    candidate = generate_report(row)
    result = score_report(row['template_content'], row['report'], candidate)
    calib_scores.append(result['RES'])
    time.sleep(SLEEP_BETWEEN_CALLS)

print(f"Mean approximate RES on {len(calib_scores)} train samples: "
      f"{sum(calib_scores)/len(calib_scores):.4f}")


## Step 2: generate test.csv predictions (with checkpointing)

Writes incrementally to a checkpoint file so a Kaggle session interruption doesn't
lose completed work -- re-running this cell resumes from the last completed case_id.


In [ ]:
test_df = pd.read_csv(TEST_CSV)
CHECKPOINT_CSV = OUTPUT_CSV + ".checkpoint"

completed = {}
if os.path.exists(CHECKPOINT_CSV):
    with open(CHECKPOINT_CSV, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            completed[r["case_id"]] = r["report"]
    print(f"Resuming: {len(completed)} cases already completed.")

fieldnames = ["case_id", "report"]
write_header = not os.path.exists(CHECKPOINT_CSV)
with open(CHECKPOINT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if write_header:
        writer.writeheader()
    for _, row in test_df.iterrows():
        cid = row["case_id"]
        if cid in completed:
            continue
        report = generate_report(row)
        writer.writerow({"case_id": cid, "report": report})
        f.flush()
        completed[cid] = report
        time.sleep(SLEEP_BETWEEN_CALLS)

print(f"Generated {len(completed)} / {len(test_df)} test reports.")


## Step 3: assemble and validate submission.csv


In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_CSV)
assert set(sample_sub['case_id']) == set(completed.keys()), \
    "Missing or extra case_ids vs sample_submission.csv"

out_df = pd.DataFrame({
    "case_id": sample_sub["case_id"],
    "report": [completed[cid] for cid in sample_sub["case_id"]],
})

bad = [cid for cid, r in zip(out_df['case_id'], out_df['report']) if not is_valid_report(r)]
if bad:
    print(f"WARNING: {len(bad)} case(s) failed format validation: {bad[:10]}")

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV} with {len(out_df)} rows.")


## Reproducibility notes

- `RULES.md` and `src/res_scorer.py` are embedded verbatim above (see
  `src/build_notebook.py`, which regenerates this notebook from those two source
  files so the prompt and the scorer can never silently drift from what was actually
  used to produce the submitted CSV).
- `MODEL`, retry, and sampling parameters are all set at the top (temperature 0 for
  determinism); no manual per-case editing occurs anywhere in this pipeline.
- No API key is stored in this notebook or in source control.
